In [1]:
import pandas as pd

In [2]:
# load datasets
orders = pd.read_csv("../data/olist_orders_dataset.csv")
items = pd.read_csv("../data/olist_order_items_dataset.csv")
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")

In [3]:
# Dates are loaded as text; convert to datetime so durations can be computed
date_cols = ["order_purchase_timestamp", "order_approved_at",
             "order_delivered_carrier_date", "order_delivered_customer_date",
             "order_estimated_delivery_date"]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"])

reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

In [4]:
# # delivery_days = days from purchase to delivery (whole days, floored)
orders["delivery_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days

In [5]:
# delay_days = delivered date minus estimated date
# Negative = delivered early, positive = delivered late
orders["delay_days"] = (orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]).dt.days

In [6]:
# is_late = True if delivered after the estimated date
# .where() keeps it empty (NaN) for undelivered orders, so they are not counted as on time
orders["is_late"] = (orders["delay_days"] > 0).where(orders["delay_days"].notnull())
orders["is_late"] = orders["is_late"].astype("boolean")

In [7]:
orders["is_late"].isnull().sum()

np.int64(2965)

In [8]:
orders["is_late"].value_counts()

is_late
False    89941
True      6535
Name: count, dtype: Int64

In [9]:

orders[orders["delay_days"] > 0][["order_delivered_customer_date", "order_estimated_delivery_date", "delay_days", "is_late"]].head()

,order_delivered_customer_date,order_estimated_delivery_date,delay_days,is_late
20,2017-10-09 22:23:46,2017-09-28,11.0,True
25,2018-03-21 22:03:54,2018-03-12,9.0,True
41,2017-12-28 18:59:23,2017-12-21,7.0,True
57,2018-04-03 13:28:46,2018-04-02,1.0,True
58,2017-04-06 13:37:16,2017-03-30,7.0,True


In [10]:
# Delay analysis sample: delivered orders that have a delivery date (96,470)
# The original orders table is left untouched for revenue analysis
delivered = orders[(orders["order_status"] == "delivered") &
                   (orders["order_delivered_customer_date"].notnull())].copy()

In [11]:
delivered.shape

(96470, 11)

In [12]:
(delivered["delivery_days"] < 0).sum()

np.int64(0)

In [13]:
delivered[["delivery_days", "delay_days"]].describe()

,delivery_days,delay_days
count,96470.000000,96470.000000
mean,12.093604,-11.875889
std,9.551380,10.182105
min,0.000000,-147.000000
25%,6.000000,-17.000000
50%,10.000000,-12.000000
75%,15.000000,-7.000000
max,209.000000,188.000000


In [14]:
delivered["is_late"].value_counts()

is_late
False    89936
True      6534
Name: count, dtype: Int64

In [15]:
# Some orders have 2-3 reviews; keep only the latest per order
# so that joins do not double count
reviews_clean = (reviews.sort_values("review_answer_timestamp")
                        .drop_duplicates(subset="order_id", keep="last")
                        .copy())

In [16]:
reviews_clean.shape

(98673, 7)

In [17]:
reviews_clean["order_id"].is_unique

True

In [18]:
reviews_clean["review_score"].value_counts()

review_score
5    57008
4    19038
1    11363
3     8133
2     3131
Name: count, dtype: int64

In [19]:
# Join delivery data with reviews on order_id
# inner join drops 646 delivered orders that have no review
analysis = delivered.merge(reviews_clean[["order_id", "review_score"]],
                           on="order_id", how="inner")

In [20]:
analysis.shape

(95824, 12)

In [21]:
analysis["order_id"].is_unique

True

In [22]:
analysis["review_score"].isnull().sum()

np.int64(0)

In [23]:
# Average review score for late vs on-time orders
analysis.groupby("is_late")["review_score"].agg(["count", "mean", "median"])

,count,mean,median
is_late,,,
False,89443,4.290386,5.0
True,6381,2.270491,1.0


In [24]:
# Group delay into buckets and compare review scores
# Bins are (lower, upper], so "1-3 days late" means delay_days of 1, 2 or 3
bins = [-float("inf"), 0, 3, 7, float("inf")]
labels = ["On time / early", "1-3 days late", "4-7 days late", "8+ days late"]
analysis["delay_bucket"] = pd.cut(analysis["delay_days"], bins=bins, labels=labels)

analysis.groupby("delay_bucket", observed=True)["review_score"].agg(["count", "mean", "median"])

,count,mean,median
delay_bucket,,,
On time / early,89443,4.290386,5.0
1-3 days late,1852,3.291037,4.0
4-7 days late,1748,2.102975,1.0
8+ days late,2781,1.696152,1.0


In [25]:
# Share of 1-star reviews in each delay bucket
(analysis.assign(one_star=analysis["review_score"] == 1)
         .groupby("delay_bucket", observed=True)["one_star"].mean() * 100)

delay_bucket
On time / early     6.618740
1-3 days late      25.107991
4-7 days late      58.581236
8+ days late       69.830996
Name: one_star, dtype: float64

In [26]:
three_plus = analysis[analysis["is_late"] == True]
print((three_plus["review_score"] == 1).sum(), (analysis["review_score"] == 1).sum())

3431 9351


## Key findings: delivery delay vs review score

- 6.7% of delivered orders (6,381 of 95,824) arrived after the estimated date.
- Late orders average 2.27 stars vs 4.29 for on-time orders.
- The score falls as delay grows: 1-3 days late averages 3.29, 4-7 days late 2.10, 8+ days late 1.70.
- 69.8% of customers whose order was 8+ days late gave 1 star, vs 6.6% for on-time orders.
- Late orders are 6.7% of reviewed deliveries but account for 36.7% of all 1-star reviews (3,431 of 9,351).
- This is a correlation. Other causes (wrong or missing items) also drive low scores.

In [27]:
import os
os.makedirs("../data/cleaned", exist_ok=True)

orders.to_csv("../data/cleaned/orders_clean.csv", index=False)
items.to_csv("../data/cleaned/items_clean.csv", index=False)
reviews_clean.to_csv("../data/cleaned/reviews_clean.csv", index=False)
analysis.to_csv("../data/cleaned/delay_review_analysis.csv", index=False)